This is a short demonstration of a Qwen finetuned model for (French) parlamentarian debates.

In [1]:
!uv pip install vllm

Using Python 3.12.11 environment at: /usr
Resolved 140 packages in 1.04s
Prepared 44 packages in 26.81s
Uninstalled 10 packages in 678ms
Installed 44 packages in 231ms
 + astor==0.8.1
 + blake3==1.0.5
 + cbor2==5.7.0
 + compressed-tensors==0.10.2
 + depyf==0.19.0
 + diskcache==5.6.3
 + dnspython==2.7.0
 + email-validator==2.3.0
 + fastapi-cli==0.0.10
 + fastapi-cloud-cli==0.1.5
 + gguf==0.17.1
 + httptools==0.6.4
 + interegular==0.3.3
 + llguidance==0.7.30
 - llvmlite==0.43.0
 + llvmlite==0.44.0
 + lm-format-enforcer==0.10.12
 + mistral-common==1.8.4
 + msgspec==0.19.0
 + ninja==1.13.0
 - numba==0.60.0
 + numba==0.61.2
 - nvidia-cudnn-cu12==9.10.2.21
 + nvidia-cudnn-cu12==9.5.1.17
 - nvidia-cusparselt-cu12==0.7.1
 + nvidia-cusparselt-cu12==0.6.3
 - nvidia-nccl-cu12==2.27.3
 + nvidia-nccl-cu12==2.26.2
 + openai-harmony==0.0.4
 + outlines-core==0.2.10
 + partial-json-parser==0.2.1.1.post6
 + prometheus-fastapi-instrumentator==7.1.0
 + pybase64==1.4.2
 + pycountry==24.6.1
 + pydantic-extr

We import the model from HuggingFace (will take about 3-4 minutes). One of the main advantage of local hosting is that models are imported once and for all.

In [2]:
from vllm import LLM, SamplingParams

llm = LLM("Pclanglais/Gemma-Discourse-2",
          max_model_len=8192)

INFO 09-04 18:45:03 [__init__.py:241] Automatically detected platform cuda.
INFO 09-04 18:45:05 [utils.py:326] non-default args: {'model': 'Pclanglais/Gemma-Discourse-2', 'max_model_len': 8192, 'disable_log_stats': True}


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

INFO 09-04 18:45:25 [__init__.py:711] Resolved architecture: Gemma3ForConditionalGeneration


...

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/548 [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/210 [00:00<?, ?B/s]

WARNING 09-04 18:45:41 [__init__.py:2921] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
INFO 09-04 18:48:40 [llm.py:298] Supported_tasks: ['generate']


We set up the sampling parameters. Since discourse generation is a creative activity, I recommend setting a high enough temperature (whereas for annotation tasks, a lower temperature might be preferable).

In [29]:
sampling_params = SamplingParams(
    temperature=0.7,           # Truly greedy decoding (deterministic)
    max_tokens=4000,
    stop=["<|im_end|>"],  # Match your EOS token
    skip_special_tokens=False,
    min_tokens = 10,
)

Now we start with a conversation in media res:

In [4]:
from pprint import pprint

prompts = ["""### Debate ###
<speech identifier="3510256" name="Paul Truc" group="Ensemble pour la République">
Enfin, nous nous appliquons maintenant à adapter des modèles de langue pour simplifier la retranscription des débats à l'assemblée. C'est, je crois, une tâche importante pour la souveraineté du pays.
</speech>

<speech identifier="3510139" name="Jules Machin" group="Ensemble pour la République">
Sauf erreur de ma part, cher collègue, votre choix s'est porté sur Qwen qui est un modèle chinois. Pour la souveraineté nous repasserons.
</speech>

<speech identifier="3510256" name="Paul Truc" group="Ensemble pour la République">
Vos sarcasmes n'ont aucune importance. Nous allons maintenant écouter Pierre-Carl Langlais pour une longue présentation de ce modèle. Pierre-Carl Langlais, vous avez la parole.
</speech>

### Speaker ###
Pierre-Carl Langlais

### Profile ###
Argumentative structure: Justification
Stance: Favorable
Debating behavior: Adherence
Deliberative quality: Justified
Deliberative quality: Responsive
Epistemic claim: Will of the people
Stance: Assertive
Emotion: Neutral

### Length ###
Long

### Analysis ###
"""]

outputs = llm.generate(prompts, sampling_params)
generated_texts = [output.outputs[0].text for output in outputs]

pprint(generated_texts[0])

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

('\n'
 '\n'
 "1. **Drafting the Argument**: Langlais's intervention is not a spontaneous "
 'reaction but the result of careful preparation. His provided '
 'features—"Justification," "Adherence," "Responsive," "Neutral"—all point to '
 'a speaker who has built his case methodically. He is not just repeating the '
 "previous speaker's point about sovereignty; he is building on it. The thread "
 "is clear: a French model (Qwen) was chosen over a Chinese one. Paul Truc's "
 'interjection was a minor challenge, but Langlais will likely frame his '
 'response as a necessary clarification or an expansion of the initial point, '
 'demonstrating his "Adherence" and "Justified" deliberative quality. He will '
 'likely frame his argument as a logical and necessary next step.\n'
 '2.  **Justification and Credibility**: The core of his speech will be to '
 'provide concrete, technical reasons why a Chinese-developed model cannot be '
 "trusted for parliamentary purposes, thereby justifying the go

Another example:

In [ ]:
prompts = ["""### Debate ###
<speech identifier="3510256" name="Paul Truc" group="Ensemble pour la République">
C'est, je crois la première fois que nous avons un débat à l'assemblée sur le prix des croquettes de chat. Le sujet peut paraître simple. Il pose en réalité de vraies questions économiques et même, je dirai géopolitique…
</speech>

<speech identifier="3510139" name="Jules Machin" group="La France Insoumise">
Mais c'est un sujet tout bonnement ridicule. Nous perdons un temps précieux.
</speech>

### Speaker ###
Paul Truc

### Profile ###
Emotion: Sarcastic
Argumentative structure: Justification
Stance: Favorable
Debating behavior: Adherence
Deliberative quality: Justified
Deliberative quality: Responsive
Epistemic claim: Will of the people
Stance: Assertive

### Length ###
Long

### Analysis ###
"""]

outputs = llm.generate(prompts, sampling_params)
generated_texts = [output.outputs[0].text for output in outputs]

pprint(generated_texts[0])

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

("Paul Truc's initial statement frames the issue of pet food prices as a "
 'complex subject with geopolitical implications, elevating a trivial matter. '
 'This is immediately met with a direct, dismissive attack from Jules Machin '
 'of La France Insoumise. Machin\'s term "ridiculous" is not an argument; it '
 'is a moral and political judgment intended to delegitimize the debate.\n'
 '\n'
 "Given Truc's profile, his response will not be a simple apology. As a member "
 'of the presidential majority ("Ensemble pour la République"), he is expected '
 "to defend the government's record and show adherence to his group's line. "
 "Machin's attack, however, likely comes from the far-left, whom the majority "
 'often frames as out of touch with economic reality.\n'
 '\n'
 "Truc's features suggest a nuanced strategy.\n"
 '*   **Favorable/Assertive:** He cannot back down. He must begin by framing '
 'the issue as a matter of "will of the people" – a core right-wing concept, '
 'connecting wi

Now in each case we're predicting only one turn. To get to multi-turn, we need to iteratively generate new conversations.

In [30]:
speaker_turn = ["Jules Truc", "Paul Machin", "Jules Truc", "Paul Machin", "Pierre-Carl Langlais", "Jules Truc", "Pierre-Carl Langlais", "Paul Machin"]

speaker_1_features = """Emotion: Sarcastic
Argumentative structure: Justification
Stance: Favorable
Debating behavior: Adherence
Deliberative quality: Justified
Deliberative quality: Responsive
Epistemic claim: Will of the people
Stance: Assertive"""

speaker_2_features = """Debating behavior: Adherence
Deliberative quality: Justified
Emotion: Concerned
Stance: Critical
Target audience: Government
Argumentative structure: Explaining the problem
Argumentative structure: Justification
Emotion: Indignant
Epistemic claim: Institutional knowledge
Epistemic claim: Practical necessity"""

speaker_3_features = """Emotion: Firm
Debating behavior: Adherence
Deliberative quality: Justified
Debating behavior: Confrontational
Argumentative structure: Justification
Target audience: Opponent
Stance: Assertive
Deliberative quality: Responsive
Stance: Challenging
Epistemic claim: Institutional knowledge
Argumentative structure: Direct accusation
Argumentative structure: Explaining the problem"""

dictionary_features = {"Jules Truc": speaker_1_features, "Paul Machin": speaker_2_features, "Pierre-Carl Langlais": speaker_3_features}
group = {"Jules Truc": "Ensemble pour la République", "Paul Machin": "France Insoumise", "Pierre-Carl Langlais": "Expert"}

seed_prompt = """### Debate ###
<speech identifier="3510256" name="Jules Truc" group="Ensemble pour la République">
C'est, je crois la première fois que nous avons un débat à l'assemblée sur le prix des croquettes de chat. Le sujet peut paraître simple. Il pose en réalité de vraies questions économiques et même, je dirai géopolitique…
</speech>"""

complete_analysis = []
complete_draft = []
complete_speech = []

start_identifier = 3510255

current_turn = 0

for current_speaker in speaker_turn[1:]:
    start_identifier = start_identifier + 1
    current_feature = dictionary_features[current_speaker]
    current_group = group[current_speaker]

    current_prompt = seed_prompt + "\n\n### Speaker ###\n" + current_speaker + "\n\n### Profile ###\n" + current_feature + "\n\n### Length ###\n"

    outputs = llm.generate([current_prompt], sampling_params)
    generated_texts = [output.outputs[0].text for output in outputs]

    generated_texts_reasoning, generated_texts_speech = generated_texts[0].split("### Speech ###\n")
    generated_analysis, generated_draft = generated_texts_reasoning.split("### Draft ###\n")

    complete_analysis.append(generated_analysis)
    complete_draft.append(generated_draft)
    complete_speech.append(generated_texts_speech)

    generated_texts_speech = '<speech identifier="' + str(start_identifier) + '" name="' + current_speaker + '" group="' + current_group + '">\n' + generated_texts_speech + "</speech>"

    seed_prompt = seed_prompt + "\n\n" + generated_texts_speech

    print(generated_texts_speech)


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

<speech identifier="3510256" name="Paul Machin" group="France Insoumise">
Debating on the price of cat food in the Assembly… is simple. What is the mechanism ?
</speech>


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

...

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

<speech identifier="3510261" name="Pierre-Carl Langlais" group="Expert">
Monsieur Machin, vous ne pouvez pas vous contenter d’évoquer un sujet de l’actualité, ce qui n’est peut-être pas un sujet de débat parlementaire. Il est également un problème industriel de l’ensemble du secteur. Je vous invite à consulter l’Institut national de la statistique et des études économiques (INSEE), qui réalise des enquêtes sur le prix des produits alimentaires pour l’homme. Dans ma circonscription, par exemple, le taux d’inflation des aliments pour chiens et pour chats, pour l’année 2023, s’élevait respectivement à 12,9 % et à 13,3 %.
</speech>


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

<speech identifier="3510262" name="Paul Machin" group="France Insoumise">
Je reviens sur le sujet que nous avons abordé au début de la séance : combien de familles de Français ne peuvent se permettre de nourrir leur animal de compagnie. Le chiffre, qui correspond à peu près à 10 % des familles, est élevé et montre que la question des prix des croquettes est devenue une grande préoccupation.
</speech>


And now we can create the full table of collected interventions:

In [31]:
import pandas as pd

df = pd.DataFrame({
    'speaker_turn': speaker_turn[1:],
    'complete_analysis': complete_analysis,
    'complete_draft': complete_draft,
    'complete_speech': complete_speech,
})

df['complete_analysis'] = df['complete_analysis'].str.replace('\n\n### Analysis ###\n', '', regex=False)

from IPython.display import display
display(df)

,speaker_turn,complete_analysis,complete_draft,complete_speech
0,Paul Machin,Jules Truc's opening speech frames the issue o...,"(Listening to Truc begin...)\n""Le premier déba...",Debating on the price of cat food in the Assem...
1,Jules Truc,Jules Truc's initial intervention sets a serio...,"(Reacting to Paul Machin's interjection)\n\n""L...","Monsieur Machin, si le sujet des croquettes de..."
2,Paul Machin,"Jules Truc, from the presidential majority, in...",(The session begins with Truc's pompous preamb...,"Le vrai problème, c’est que des familles doive..."
3,Pierre-Carl Langlais,"The debate has been framed by Jules Truc, who ...","(Listening to the exchange)\n\nAlright, Truc i...",Regardez-vous ! Machin a raison de dire que le...
4,Jules Truc,The debate has veered away from Jules Truc's o...,"(internal monologue)\n\nAlright, this is going...","Vous m’avez donné l’occasion, monsieur Machin,..."
5,Pierre-Carl Langlais,The debate revolves around the rising cost of ...,(Listening to Machin)\nHe's not wrong. Of cour...,"Monsieur Machin, vous ne pouvez pas vous conte..."
6,Paul Machin,The debate has reached a point of procedural a...,(Stream of consciousness)\n\nThey're doing it ...,Je reviens sur le sujet que nous avons abordé ...
